# 3장 언어 모델의 추론

프롬프트를 이용한 추론과 모델 자체의 추론을 실습합니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 3장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다. 원격 모델 API를 사용한다면 Ollama 설치 셀은 건너뜁니다.

In [ ]:
# %%capture
# !pip install illustrated-agents

Colab 안에서 Ollama로 모델을 실행하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택한 뒤, 아래 셀의 주석을 해제하고 실행합니다.

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b && ollama pull gemma3:12b &

## 언어 모델 선택

이번 실습에 사용할 언어 모델을 지정합니다.

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 3 12B(내장 추론과 도구 호출 없음)
llm = LLM(model="gemma3:12b", temperature=1)

다른 언어 모델을 사용하려면 아래에서 로컬 서버 또는 클라우드 API에 해당하는 설정을 선택합니다.

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-3-12B-it-Q4_K_M", base_url="http://127.0.0.1:8080", temperature=1)

# # LM Studio
# llm = LLM(model="gemma-3-12b-it", base_url="http://127.0.0.1:1234/v1", temperature=1)

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-3-12b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"], temperature=1)

`temperature`를 1로 설정하면 같은 입력에서도 서로 다른 답을 표본으로 얻을 수 있습니다. 0에 가깝게 설정하면 같은 답이 반복될 가능성이 높습니다. 수수께끼를 제시하기 전에 짧은 답변 예시를 보여 주고 모델이 그 형식을 따르는지 살펴봅니다. 이처럼 소수의 예시로 원하는 동작을 알려 주는 방법을 퓨샷 학습이라고 합니다.

---

## 1. 사고의 연쇄(Chain-of-Thought, CoT)

모델은 학습을 통해 추론하도록 만들 수 있지만, 추가 학습에는 시간과 GPU 연산이 필요합니다. 여기서는 별도의 학습 없이 프롬프트만 바꿔 중간 풀이를 생성하도록 합니다. `TinyAgent`에 추론을 통합하는 과정은 6장에서 다룹니다.

![3장의 추론 실습과 TinyAgent의 구성](../images/ch3.png)

먼저 사용자 질문과 답변 예시를 한 쌍(원샷) 또는 몇 쌍(퓨샷) 보여 줍니다. 짧은 답만 있는 예시를 본 모델이 새로운 문제에도 같은 형식으로 답하는지 확인합니다.

아래 예시 질문은 홍학 여섯 마리 중 두 마리가 날아가고 한 마리가 나무 뒤에 숨으면 몇 마리가 보이는지 묻습니다. 실제로 풀 질문에서는 펭귄 아홉 마리 중 두 마리가 물속으로 사라지고 네 마리가 해안에서 걸어옵니다.

In [ ]:
from rich import print

# 퓨샷 프롬프트 예시
prompt_example = "I saw 6 flamingos. 2 flew away. 1 hid behind a tree. How many can I see? Be concise."
prompt_answer = "3"

# 모델에 보낼 질문
query = "I saw 9 penquins. 2 slid in the water and disappeared from sight whilst 4 waddled up from the shore. How many can I see?"

# 메시지 목록
messages = [
    {"role": "user", "content": prompt_example},
    {"role": "assistant", "content": prompt_answer},
    {"role": "user", "content": query},
]

# 응답 생성
response = llm.generate(messages)
print(response)

이 예에서는 모델이 풀이 없이 답을 냈고 그 답도 틀렸습니다. 이제 `prompt_answer`에 답을 구하는 과정까지 적습니다. 모델이 이 예시를 따르면 새로운 질문에도 풀이를 쓴 뒤 최종 답변을 제시합니다.

In [ ]:
prompt_example = "I saw 6 flamingos. 2 flew away. 1 hid behind a tree. How many can I see?"
prompt_answer = "You saw 6 flamingos 2 flew away. 6 - 2 = 4. Then, 1 flamingo hid. You see now 4 - 1 = 3."

# 메시지 목록
messages = [
    {"role": "user", "content": prompt_example},
    {"role": "assistant", "content": prompt_answer},
    {"role": "user", "content": query},
]

# 응답 생성
response = llm.generate(messages)
print(response)

저자의 실행에서는 정답을 얻었습니다. 최종 답을 쓰기 전에 어떤 풀이를 생성했는지 확인합니다.

## 2. 예시 없이 추론 유도하기(Zero-shot CoT)

질문에 맞는 풀이 예시를 준비하려면 시간이 들고 입력 토큰도 늘어납니다. 예시 없이 “Let's think step by step”(단계별로 생각해 보자)이라는 지시를 추가할 수도 있습니다. 학습 중에 풀이 과정을 충분히 접한 모델은 이 요청만으로 더 자세한 풀이를 생성하기도 합니다.

In [ ]:
# 제로샷 CoT 지시가 포함된 메시지
messages = [{"role": "user", "content": query + " Let's think step by step."}]

# 응답 생성
response = llm.generate(messages)
print(response)

저자의 실행에서는 이 지시를 추가했을 때 풀이가 길어졌고 정답이 나왔습니다.

`TinyAgent`에 같은 제로샷 CoT 지시를 적용해 봅니다.

In [ ]:
from illustrated_agents.chapters.ch2 import TinyAgent

agent = TinyAgent(llm=llm)
response = agent.run(query + " Let's think step by step.")

In [ ]:
print(response)

## 3. 자기 일관성(Self-Consistency)

하나의 질문에 대해 여러 답을 생성하고 가장 자주 나온 답을 선택합니다. 풀이가 다양해지도록 일반적으로 높은 `temperature`와 CoT 프롬프트를 함께 사용합니다.

좌석 배치 문제를 모델에 열 번 묻습니다. 실행할 때마다 풀이와 답이 달라질 수 있습니다. “Answer:” 뒤에 최종 답을 쓰도록 지정하면 풀이에서 답을 분리해 각 답이 나온 횟수를 셀 수 있습니다.

문제의 조건은 여섯 사람이 한 줄에 앉고, Sarah는 3번, Chris는 양 끝 중 한 곳에 앉는다는 것입니다. Jor는 Chris의 바로 오른쪽에, Ilse는 Sarah와 Maarten의 정확한 가운데에 앉습니다. Irene은 Sarah 옆에 앉지 않고 Maarten은 Irene보다 왼쪽에 있습니다. 이 조건에서 Maarten의 좌석 번호를 구합니다.

모델을 열 번 호출하므로 하드웨어에 따라 시간이 걸릴 수 있습니다. 이 셀은 건너뛸 수 있습니다.

In [ ]:
from collections import Counter

# 자기 일관성 실험에서 다양한 답을 얻기 위해 temperature=1 사용
llm = LLM(model="gemma3:12b", temperature=1)

answers = []
for _ in range(10):
    query = """Six friends (Maarten, Ilse, Sarah, Jor, Irene, and Chris) are sitting in a row.

- Sarah is in seat 3.
- Chris is sitting at one of the ends of the row.
- Jor is sitting immediately to the right of Chris.
- Ilse is sitting exactly in the middle of Sarah and Maarten.
- Irene is not sitting next to Sarah.
- Maarten is sitting somewhere to the left of Irene.

In which seat is Maarten sitting?
Let's think step by step and give back your answer only after 'Answer:'.
"""

    # 응답 생성
    messages = [{"role": "user", "content": query},]
    response = llm.generate(messages)
    answer = response.content.split("Answer:")[-1].replace("\n", "").strip()
    answers.append(answer)

Counter(answers)

이 문제의 정답은 5번 좌석입니다. 저자들의 실험에서는 1번과 5번이 자주 나왔습니다. `temperature`가 1이므로 실행할 때마다 득표수가 달라지고, 오답이 과반을 차지할 수도 있습니다. 다른 질문으로도 시도해 보세요. 모델이 정답을 낼 확률이 낮으면 호출 비용이 커져도 다수결의 정확도가 충분히 높아지지 않을 수 있습니다.

## 4. 여러 후보 중 선택하기(Best-of-N)

모델이 답변 후보 N개를 생성하면 검증기가 각각을 채점하고 점수가 가장 높은 답변을 선택합니다. 후보를 다양하게 얻기 위해 높은 온도를 쓰거나 온도를 바꾸며 생성합니다.

예제에서는 로마 숫자를 정수로 변환하는 함수(예: IV → 4)를 작성하게 합니다. 입력과 기대 결과가 정해진 테스트를 준비하고 통과한 개수로 각 후보 함수를 평가합니다.

생성할 함수는 1~3999 범위에서 IV, IX, XL, XC, CD, CM 같은 뺄셈 표기를 처리해야 합니다. 빈 문자열이나 잘못된 문자·반복 패턴에는 0을 반환합니다. 프롬프트는 설명과 마크다운 없이 함수 정의만 출력하도록 요구합니다.

In [ ]:
test_cases = [
    ("III", 3),
    ("IV", 4),
    ("IX", 9),
    ("LVIII", 58),
    ("MCMXCIV", 1994),
    ("MMMCMXCIX", 3999),
    ("", 0),
    ("IIII", 0),       # 잘못된 표기: 같은 기호가 네 번 연속 등장
    ("VV", 0),         # 잘못된 표기: V는 반복할 수 없음
    ("IC", 0),         # 잘못된 표기: I는 V 또는 X 앞에서만 뺄 수 있음
    ("ABC", 0),        # 잘못된 표기: 로마 숫자 이외의 문자
    ("MMMM", 0),       # 범위 초과: 3999보다 큰 값
]


def verify(answer):
    try:
        # 함수 추출
        namespace = {}
        exec(answer, namespace)
        fn = namespace["roman_to_int"]

        # 테스트 사례로 함수 평가
        score = sum(fn(inp) == expected for inp, expected in test_cases) / len(test_cases)
        return score
    except Exception:
        # 유효한 파이썬 코드가 아니면 0점 반환
        return 0.0

앞의 자기 일관성 실습처럼 여러 번 생성하되, 이번에는 테스트 결과를 기준으로 후보를 선택합니다.

모델을 열 번 호출합니다. 실행 시간이 길면 이 셀을 건너뛸 수 있습니다.

In [ ]:
responses = []
for _ in range(10):
    # 모델에 보낼 질문
    query = """Write a Python function `roman_to_int(s)` that converts a Roman numeral 
string to an integer. The function should:
- Handle standard Roman numerals from 1 to 3999.
- Correctly handle subtractive notation (e.g., IV=4, IX=9, XL=40, XC=90, CD=400, CM=900).
- Return 0 for invalid inputs (empty string, non-Roman characters, or invalid patterns 
  like "IIII" or "VV").

Give only the function definition, no explanation or markdown formatting."""

    # 응답 생성
    response = llm.generate([{"role": "user", "content": query}])

    # 응답 검증 및 점수 계산
    score = verify(response.content)

    # 응답과 점수 저장
    responses.append((response.content, score))

# 최고 점수와 해당 답변 선택
best_answer, best_score = max(responses, key=lambda x: x[1])

# 모든 점수 표시
[response[1] for response in responses]

저자의 실행에서 모든 테스트를 통과한 답은 하나였습니다. 한 번만 생성했을 때 놓칠 수 있는 답을 여러 번 생성하고 검증해 찾아낸 예입니다.

## 5. 모델 자체의 추론

일부 모델은 프롬프트에서 별도로 풀이를 요구하지 않아도 추론 내용을 출력합니다. 학습할 때 추론에 특수 토큰을 사용했기 때문입니다. 예제 모델의 대화 템플릿에서 시스템 메시지와 사용자 메시지, 추론을 요청하는 토큰이 배치된 위치를 확인합니다.

<pre style="background:#1e1e1e; color:#d4d4d4; padding:16px; border-radius:8px; overflow-x:auto; font-size:13px; line-height:1.8;">
<span style="color:#f97583;">&lt;|turn&gt;system</span>
<span style="color:#4DDC52;">&lt;|think|&gt;</span><span style="background:#D7D7D7; color:#000000; padding:1px 6px; border-radius:3px;">SYSTEM_PROMPT</span><span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;user</span>
<span style="background:#D7D7D7; color:#000000; padding:1px 6px; border-radius:3px;">USER_PROMPT</span><span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;model</span>
</pre>

이 템플릿을 지원하는 Gemma 4 E4B를 불러옵니다.

In [ ]:
# OpenAI 호환 API로 Ollama 호출
llm = LLM(model="gemma4:e2b", think=True)

추론 엔진이 메시지에 대화 템플릿을 적용합니다. 예제에서는 Ollama에 추론을 켜 달라고 지정하면 엔진이 `<|think|>` 토큰을 추가합니다.

In [ ]:
# 모델에 보낼 질문
query = "I saw 9 penquins. 2 slid in the water and disappeared from sight whilst 4 waddled up from the shore. How many can I see?"

# 응답 생성
response = llm.generate([{"role": "user", "content": query}])
print(response)

응답의 `reasoning` 필드에서 추론 내용을 바로 확인할 수 있습니다. 예전에는 호출자가 모델별 템플릿을 정확하게 작성해야 했지만, 여기서는 추론 서비스가 템플릿 적용과 응답 분리를 담당합니다.

---

# 실습한 내용

이번 장에서는 프롬프트를 바꿔 Gemma 3 12B가 풀이를 생성하도록 했습니다. Gemma 4의 내장 추론은 2장에서 구현한 `think` 매개변수로 제어합니다. 두 방식 모두 기존 `TinyAgent` 클래스로 실습할 수 있습니다.

# 다음 장

[4장](../chapter04/chapter04.ipynb)에서는 이전 대화를 다음 요청에 사용할 수 있도록 메모리를 구현합니다.